# portao-qualidade.ipynb — confere o vídeo ANTES de publicar

Existe por causa de uma família de bugs que o pipeline tem e que **não dá erro
nenhum**: aparece só no vídeo pronto, às vezes só depois de publicado.

Dois casos reais que já aconteceram neste projeto:

- um idioma novo caiu no **cinza de fallback** em vez da cor da paleta
- a legenda saiu em **quadradinhos** porque a fonte não tinha os glifos do idioma

Nenhum dos dois levantou exceção. Os dois são triviais de detectar antes.

## Duas camadas

| Quando | O que confere | Custo |
|---|---|---|
| **Antes de queimar** (`.ass`) | cor fora da paleta, fonte sem glifo, área segura, tempo | milissegundos |
| **Depois de queimar** (`.mp4`) | áudio estourado, duração, faixa de áudio | ~1 min |

Rode a primeira **sempre** — ela é de graça e pega a maior parte. A segunda,
antes de subir.

## Sobre a fonte

O projeto declara `Arial` no estilo padrão, e Linux/Colab não tem Arial: o
fontconfig troca por Liberation Sans, que é metricamente compatível e cobre
latim inteiro. Isso é **aviso, não erro** — se fosse erro, o portão gritaria em
todo vídeo e você aprenderia a ignorá-lo, o que é pior do que não ter portão.

O que é erro sempre: **glifo faltando**. É o que vira quadradinho na tela.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP                                                         ║
# ╚══════════════════════════════════════════════════════════════════╝

# fontTools lê a tabela de glifos da fonte -- sem ele, a checagem mais
# importante (glifo faltando) não roda.
!pip install -q fonttools
print('✅ fonttools')

# Fontes CJK. Sem elas, declarar "Noto Sans CJK SC" no .ass não adianta:
# o fontconfig cai em outra fonte e o coreano/chinês vira quadradinho.
!apt-get -qq -y install fonts-noto-cjk > /dev/null 2>&1
!fc-cache -f > /dev/null 2>&1
print('✅ fonts-noto-cjk')

from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

import shutil, sys
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO_MODULOS = Path("/content/pipeline")

if DESTINO_MODULOS.exists():
    shutil.rmtree(DESTINO_MODULOS)
shutil.copytree(PASTA_MODULOS, DESTINO_MODULOS)
if str(DESTINO_MODULOS) not in sys.path:
    sys.path.insert(0, str(DESTINO_MODULOS))
print(f"✅ {len(list(DESTINO_MODULOS.glob('*.py')))} módulos")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO — edite só esta célula                          ║
# ╚══════════════════════════════════════════════════════════════════╝

NOME_ORACAO = "40_Matt_02"

# ── Quais arquivos conferir ─────────────────────────────────────────────────
# Deixe "" pra pular. Os nomes seguem o padrão do config.py.
ARQUIVO_ASS   = f"legendas_{NOME_ORACAO}_v2.ass"
ARQUIVO_VIDEO = f"{NOME_ORACAO}_final_multicolor.mp4"

# ── Margem da área segura ───────────────────────────────────────────────────
# 5% de cada borda. TV antiga cortava mais; celular e web, quase nada -- mas a
# barra de progresso do player cobre o rodapé, então margem embaixo importa.
MARGEM_SEGURA = 0.05

# ── Duração esperada (opcional) ─────────────────────────────────────────────
# Deixe None pra não conferir. Serve pra pegar concatenação incompleta ou
# legenda que termina antes do áudio.
DURACAO_ESPERADA_MS = None

print(f"Vídeo .... {NOME_ORACAO}")
print(f"ASS ...... {ARQUIVO_ASS or '(pulando)'}")
print(f"MP4 ...... {ARQUIVO_VIDEO or '(pulando)'}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INICIALIZAR                                                   ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
from pathlib import Path

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

import qualidade as q

PASTA = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/videos/{NOME_ORACAO}")
if not PASTA.exists():
    raise SystemExit(f"❌ Pasta do vídeo não existe: {PASTA}")

print(f"📁 {PASTA}")
for nome in (ARQUIVO_ASS, ARQUIVO_VIDEO):
    if nome:
        caminho = PASTA / nome
        marca = "✅" if caminho.exists() else "❌"
        tam = f"{caminho.stat().st_size/1e6:.1f} MB" if caminho.exists() else "não existe"
        print(f"   {marca} {nome} — {tam}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 1/2 — LEGENDA (.ass)  ·  antes de queimar                     ║
# ╚══════════════════════════════════════════════════════════════════╝

rel_ass = None
if ARQUIVO_ASS and (PASTA / ARQUIVO_ASS).exists():
    rel_ass = q.verificar_ass(PASTA / ARQUIVO_ASS, margem_segura=MARGEM_SEGURA)
    print(rel_ass)
    print()
    print(f"{'✅ APROVADO' if rel_ass.aprovado else '❌ REPROVADO'}"
          f"   ({len(rel_ass.erros)} erro, {len(rel_ass.avisos)} aviso)")
else:
    print("⏭️  sem .ass pra conferir")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎬 2/2 — VÍDEO (.mp4)  ·  antes de publicar                      ║
# ║  O volumedetect lê o arquivo inteiro — leva ~1 min                ║
# ╚══════════════════════════════════════════════════════════════════╝

rel_video = None
if ARQUIVO_VIDEO and (PASTA / ARQUIVO_VIDEO).exists():
    rel_video = q.verificar_video(PASTA / ARQUIVO_VIDEO,
                                  duracao_esperada_ms=DURACAO_ESPERADA_MS)
    print(rel_video)
    print()
    print(f"{'✅ APROVADO' if rel_video.aprovado else '❌ REPROVADO'}"
          f"   ({len(rel_video.erros)} erro, {len(rel_video.avisos)} aviso)")
else:
    print("⏭️  sem .mp4 pra conferir")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🏁 VEREDITO                                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

relatorios = [r for r in (rel_ass, rel_video) if r is not None]
erros  = [a for r in relatorios for a in r.erros]
avisos = [a for r in relatorios for a in r.avisos]

print("=" * 66)
if not relatorios:
    print("⏭️  Nada foi conferido — confira os nomes na Configuração.")
elif erros:
    print(f"❌ NÃO PUBLIQUE — {len(erros)} erro(s)")
    print()
    for a in erros:
        print(str(a).replace("\n", "\n"))
    print()
    print("Erro aqui é coisa que o espectador vê: cor errada, quadradinho no")
    print("lugar da letra, áudio distorcido. Conserte e remonte.")
else:
    print(f"✅ PODE PUBLICAR   ({len(avisos)} aviso)")
    if avisos:
        print()
        for a in avisos:
            print(str(a).replace("\n", "\n"))
        print()
        print("Aviso não impede — é pra você saber, não pra travar.")
print("=" * 66)